# CUAD — Explainable Contract Review Priorities

## Project objective

Our BERT model predicts which CUAD clause categories appear in a contract. **This notebook adds a separate, rule-based review-priority layer** that looks for particular phrases inside contract text and shows the evidence behind each flag.

**Workflow:** contract text → candidate clause categories (BERT, when available) + transparent text rules → Low / Medium / High **review priority** → human assessment.

**Important distinction:** A clause category is *not* a legal-risk label. A contract can contain a `Termination For Convenience` clause without that clause being unfavorable. We therefore flag **potential review issues**, not verified legal defects. The prototype cannot replace a legal reviewer.

This notebook uses only standard Python for its rule tests. **It does not retrain BERT**, and it can be prepared while the model-validation notebook is running. The optional model-inference section is disabled by default.

## 1. Method and decisions

| Term | Meaning in our implementation |
|---|---|
| **Rule-based scoring** | A written condition triggers a review flag; the condition and matched text are visible |
| **Evidence** | The exact phrase and nearby text that caused the flag |
| **High priority** | A strong *potential* concern appears; a reviewer should inspect it first |
| **Medium priority** | A term may require further context, negotiation, or policy checks |
| **Low priority** | None of our **implemented patterns** matched; **this does not mean the contract is safe** |
| **Aggregation rule** | A contract gets its highest triggered review priority; every flag stays visible |

### Safeguards

1. **Clause existence alone does not imply risk.** We look for specific text, not just the BERT category name.
2. **Do not treat missing clauses as safe.** Unmatched rules produce *low review priority under this limited rule set*, not a finding of low legal risk.
3. **Keep evidence.** Each flag includes the rule, exact matched words, and surrounding excerpt.
4. **Do not silently discard model misses.** The text rules can run independently of BERT; a model score is added later when available.
5. **Human confirmation is required.** Legal meaning depends on which party bears the obligation, exceptions, governing law, and business context.

### Initial rule scope

These are **illustrative research rules**, not legal standards and not confirmed Accenture policy. They cover a small subset of CUAD's 41 categories. We will extend or change them based on advisor-reviewed examples and evaluate false positives and false negatives before claiming useful real-world performance.

## 2. Define the rule library

Each rule has a short identifier, CUAD clause category, regex pattern, review priority, and a one-line reason. A regex means a *text pattern*; matching it creates a **candidate flag for review**, not an automatic legal conclusion.

| Rule | Category | Potential issue we flag |
|---|---|---|
| `LIAB_UNLIMITED` | Cap On Liability | Wording that appears to allow unlimited liability |
| `TERM_NO_NOTICE` | Termination For Convenience | Termination language combined with no notice |
| `INDEMNITY_UNLIMITED` | Indemnification | Indemnity obligations described as unlimited |
| `EXCL_BROAD` | Exclusivity | Worldwide or perpetual exclusivity language |
| `RENEWAL_AUTO` | Renewal Term | Automatic renewal wording, which may need a cancellation review |
| `CHANGE_CONTROL` | Change Of Control | Change-of-control wording combined with acceleration or immediate termination |

**Why no automatic Governing Law risk score?** A jurisdiction's presence alone does not tell us whether the clause is unfavorable. We need company-specific policy and legal context before scoring it.

In [1]:
import json
import re
from pathlib import Path

RULES = [
    {
        "id": "LIAB_UNLIMITED", "category": "Cap On Liability", "priority": "High",
        "pattern": r"\bunlimited\s+liability\b|\bno\s+(?:cap|limit|limitation)\s+(?:on|of|to)\s+liability\b",
        "reason": "Possible unlimited liability; review scope, exceptions, and which party is exposed."
    },
    {
        "id": "TERM_NO_NOTICE", "category": "Termination For Convenience", "priority": "High",
        "pattern": r"\bterminat(?:e|ed|ion)\b.{0,80}\bwithout\s+(?:prior\s+)?notice\b",
        "reason": "Termination without notice may require closer review of parties and exceptions."
    },
    {
        "id": "INDEMNITY_UNLIMITED", "category": "Indemnification", "priority": "High",
        "pattern": r"\bindemnif\w*\b.{0,90}\bwithout\s+limitation\b|\bunlimited\s+indemnif\w*\b",
        "reason": "Potentially unlimited indemnity obligation; review allocation of responsibility."
    },
    {
        "id": "EXCL_BROAD", "category": "Exclusivity", "priority": "Medium",
        "pattern": r"\b(?:worldwide|perpetual)\b.{0,70}\bexclusive\b|\bexclusive\b.{0,70}\b(?:worldwide|perpetual)\b",
        "reason": "Broad exclusivity might restrict business activity; review duration and geography."
    },
    {
        "id": "RENEWAL_AUTO", "category": "Renewal Term", "priority": "Medium",
        "pattern": r"\bautomatically\s+renew\b|\bautomatic\s+renewal\b",
        "reason": "Automatic renewal may require reviewing notice and cancellation requirements."
    },
    {
        "id": "CHANGE_CONTROL", "category": "Change Of Control", "priority": "Medium",
        "pattern": r"\bchange\s+of\s+control\b.{0,90}\b(?:immediate\s+termination|accelerat(?:ion|e))\b",
        "reason": "Change-of-control consequences may need business and legal review."
    }
]

PRIORITY_ORDER = {"Low": 0, "Medium": 1, "High": 2}
for rule in RULES:
    rule["compiled"] = re.compile(rule["pattern"], re.IGNORECASE | re.DOTALL)

print("Rules loaded:", len(RULES))
print("CUAD categories currently covered:", len({r["category"] for r in RULES}))

Rules loaded: 6
CUAD categories currently covered: 6


## 3. Find evidence and assign provisional review priorities

We scan the source contract for each rule. A flag includes the **exact match**, **character location**, and a short **supporting excerpt**. If multiple rules match, we retain all findings and use the highest priority for sorting contracts.

To avoid treating clearly negated language as a definite high-priority match, a possible negation near a matched phrase is downgraded to **Medium** and marked for context review. This is only a basic safeguard; full legal negation and exceptions are more complex than keyword matching.

In [2]:
def context_excerpt(text, start, end, side=100):
    left = max(0, start - side)
    right = min(len(text), end + side)
    return " ".join(text[left:right].split())


def nearby_negation(text, start):
    # Conservative warning, not a complete linguistic negation detector.
    before = text[max(0, start - 45):start].lower()
    return bool(re.search(r"\b(?:not|never|shall not|may not)\b[^.;\n]{0,40}$", before))


def scan_contract(contract_text, contract_id="sample-contract"):
    findings = []
    for rule in RULES:
        for match in rule["compiled"].finditer(contract_text):
            needs_context = nearby_negation(contract_text, match.start())
            priority = ("Medium" if needs_context and rule["priority"] == "High"
                        else rule["priority"])
            findings.append({
                "contract_id": contract_id,
                "rule_id": rule["id"],
                "category": rule["category"],
                "review_priority": priority,
                "pattern_priority": rule["priority"],
                "matched_text": match.group(0),
                "char_start": match.start(),
                "char_end": match.end(),
                "evidence_excerpt": context_excerpt(contract_text, match.start(), match.end()),
                "reason": rule["reason"],
                "context_warning": needs_context,
                "human_review_required": True
            })
    # Prioritize the strongest signal, and keep source order for ties.
    findings.sort(key=lambda x: (-PRIORITY_ORDER[x["review_priority"]], x["char_start"]))
    priority = findings[0]["review_priority"] if findings else "Low"
    return {
        "contract_id": contract_id,
        "review_priority": priority,
        "priority_interpretation": (
            "Highest priority among matched research rules; not a legal conclusion."
            if findings else
            "No implemented rule matched; NOT an assessment that this contract is safe."
        ),
        "rule_set_size": len(RULES),
        "categories_assessed": sorted({r["category"] for r in RULES}),
        "findings": findings,
        "human_review_required": True
    }


def print_review(report):
    print("Contract:", report["contract_id"])
    print("Review priority:", report["review_priority"])
    print("Interpretation:", report["priority_interpretation"])
    print("Flags:", len(report["findings"]))
    for item in report["findings"]:
        print(f" - {item['review_priority']} | {item['rule_id']} | {item['category']}")
        print("   Evidence:", item["evidence_excerpt"])
        if item["context_warning"]:
            print("   CAUTION: Nearby negation; check full wording.")

## 4. Test the rules on synthetic contracts

We start with **made-up examples**, not actual CUAD validation/test contracts. This verifies that the rule code runs and produces readable evidence. These tests **do not measure risk-detection accuracy**. We must later evaluate against reviewer-labeled contract excerpts.

The tests run with standard Python only, so they do not load BERT or use the GPU.

In [3]:
EXAMPLES = {
    "example_high": (
        "The supplier accepts unlimited liability for damages resulting from breach. "
        "The agreement may be terminated without prior notice by the buyer."
    ),
    "example_medium": (
        "This license grants worldwide exclusive distribution rights. "
        "The agreement will automatically renew for another year unless canceled."
    ),
    "example_no_trigger": (
        "This agreement is governed by the laws stated herein. "
        "Payment must be made within thirty days."
    ),
    "example_negated": "The supplier shall not terminate this contract without notice."
}

TEST_REPORTS = {name: scan_contract(text, name) for name, text in EXAMPLES.items()}
assert TEST_REPORTS["example_high"]["review_priority"] == "High"
assert TEST_REPORTS["example_medium"]["review_priority"] == "Medium"
assert TEST_REPORTS["example_no_trigger"]["review_priority"] == "Low"
assert TEST_REPORTS["example_negated"]["review_priority"] != "High"

for name, report in TEST_REPORTS.items():
    print("\n" + "=" * 65)
    print_review(report)
print("\nSynthetic rule tests passed.")


Contract: example_high
Review priority: High
Interpretation: Highest priority among matched research rules; not a legal conclusion.
Flags: 2
 - High | LIAB_UNLIMITED | Cap On Liability
   Evidence: The supplier accepts unlimited liability for damages resulting from breach. The agreement may be terminated without prior notice by the buye
 - High | TERM_NO_NOTICE | Termination For Convenience
   Evidence: The supplier accepts unlimited liability for damages resulting from breach. The agreement may be terminated without prior notice by the buyer.

Contract: example_medium
Review priority: Medium
Interpretation: Highest priority among matched research rules; not a legal conclusion.
Flags: 2
 - Medium | EXCL_BROAD | Exclusivity
   Evidence: This license grants worldwide exclusive distribution rights. The agreement will automatically renew for another year unless canceled.
 - Medium | RENEWAL_AUTO | Renewal Term
   Evidence: This license grants worldwide exclusive distribution rights. The a

## 5. Optional: connect to the saved BERT checkpoint (no training)

Once `04_CUAD_BERT_Model.ipynb` has saved the model, we can load that checkpoint to predict CUAD clause categories in a **new** contract.

**Important modeling limitation:** Our BERT classifier predicts categories for 512-token **windows**; it does *not* pinpoint the exact clause boundaries. Therefore, we continue to use the actual matched rule text as evidence, and we do not pretend that a BERT window is a precisely extracted legal clause.

Our first prototype deliberately scans the **whole contract** with the transparent rules, even when BERT does not predict a category. This avoids hiding an explicit text trigger solely because the model missed a category. Later, BERT's confidence can be displayed separately as *supporting information*, never as proof of legal risk.

The optional code below is **disabled**. Do not enable it while the other notebook is still busy validating or before the model checkpoint exists. When enabled, it loads **saved weights**, not the training code.

In [4]:
RUN_SAVED_MODEL_DEMO = False  # Set True later, only after the model is saved.

if RUN_SAVED_MODEL_DEMO:
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    MODEL_DIR = Path("../data/processed/bert_clause_model_v4/full_train")
    checkpoint = MODEL_DIR / "model.safetensors"
    if not checkpoint.exists():
        raise FileNotFoundError(f"Saved model not found: {checkpoint}. Finish Section 6 of notebook 04 first.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.eval()
    # CPU keeps this small demo away from the GPU immediately after a long training run.
    device = torch.device("cpu")
    model.to(device)

    settings_path = MODEL_DIR / "training_report.json"
    settings = json.loads(settings_path.read_text(encoding="utf-8"))
    threshold = float(settings["selected_threshold"])

    SAMPLE_CONTRACT = EXAMPLES["example_high"]
    encoded = tokenizer(
        SAMPLE_CONTRACT, max_length=512, truncation=True, stride=256,
        return_overflowing_tokens=True, padding=False
    )
    # For very long contracts, process windows in batches to keep memory low.
    window_scores = []
    with torch.inference_mode():
        for start in range(0, len(encoded["input_ids"]), 4):
            windows = []
            for i in range(start, min(start + 4, len(encoded["input_ids"]))):
                row = {"input_ids": encoded["input_ids"][i],
                       "attention_mask": encoded["attention_mask"][i]}
                if "token_type_ids" in encoded:
                    row["token_type_ids"] = encoded["token_type_ids"][i]
                windows.append(row)
            batch = tokenizer.pad(windows, return_tensors="pt")
            scores = torch.sigmoid(model(**batch).logits).cpu().numpy()
            window_scores.extend(scores)

    max_scores = np.max(np.stack(window_scores), axis=0)
    candidate_categories = [
        {"category": model.config.id2label[i], "confidence": round(float(score), 4)}
        for i, score in enumerate(max_scores) if score >= threshold
    ]
    print("Saved-model category candidates (not risk conclusions):")
    for candidate in sorted(candidate_categories, key=lambda item: -item["confidence"]):
        print(candidate)
    print("\nRule-based review flags (based on explicit text evidence):")
    print_review(scan_contract(SAMPLE_CONTRACT, "model_demo"))
else:
    print("Model loading skipped. Rule tests above can run without BERT or a GPU.")

Model loading skipped. Rule tests above can run without BERT or a GPU.


## 6. Save the prototype report

We save the **synthetic test report** as JSON to make the output easy to inspect. It is labeled as a demo and is **not** a report about a real CUAD contract. Running this cell does not involve BERT.

Later, we can call `scan_contract(real_contract_text, contract_id)` and save the result using the same structure.

In [5]:
OUTPUT_DIR = Path("../data/processed/risk_review_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export = {
    "report_type": "synthetic_demonstration_only",
    "rule_method": "transparent_regex_review_flags",
    "not_legal_advice": True,
    "not_model_accuracy": True,
    "reports": TEST_REPORTS
}
file_path = OUTPUT_DIR / "synthetic_risk_demo.json"
file_path.write_text(json.dumps(export, indent=2), encoding="utf-8")
print("Saved:", file_path.resolve())

Saved: /Users/rashipatel/Documents/Accenture Project /Accenture-1G-contract-review-challenge/data/processed/risk_review_v1/synthetic_risk_demo.json


## 7. Conclusion and next checks

### What we implemented

We created a **transparent, evidence-based review-priority prototype**. Each flag identifies its rule, candidate CUAD category, matched phrase, supporting excerpt, and preliminary priority. A contract receives the highest priority triggered by the **implemented** rules, but the output does not claim a legally validated risk rating.

### What we still need to validate

- **Check rule accuracy:** Apply the rules to a small set of contracts/excerpts **reviewed by an advisor or qualified reviewer**. Count false alarms and missed concerns. Synthetic examples do not establish accuracy.
- **Check category performance:** Review BERT's category-level validation report. The clause classifier's overall F1 does not measure risk-scoring accuracy.
- **Evaluate realistic text:** Test unusual formatting, negations, exceptions, references to other sections, and which party bears an obligation.
- **Broaden rule coverage:** Our current six categories are only a starting set. Other CUAD categories are not automatically assessed.
- **Finalize end-to-end review:** Load saved BERT weights in a separate prediction session, use the rule evidence, and let a human verify each flag.

### One-sentence presentation explanation

*We combined BERT clause detection with transparent text-based review rules, so the system can highlight potential concerns and show the supporting contract wording for human evaluation.*

**Current status:** Rule prototype implemented and synthetic tests passed; legal-risk accuracy **not yet measured**.